# Enveda CASMI 2026 — local/Colab workbench

**Workbench version 0.5.0 · 2026-11-19 UTC**

This notebook consolidates the competition contract, data audit, identity/MRR tests, preprocessing smoke checks and an explicitly bounded experimental PyTorch run. It is designed for Google Colab or local Jupyter after extracting `casmi_project_v0.5.0.zip`.

> **Scientific warning:** The Kaggle-visible `test.parquet` is a training-derived preview and is replaced by hidden data on Kaggle rerun. Never use it as independent validation. The current PyTorch pipeline is a prototype, not an MRR-validated final solution; the smoke run below is exploratory only. Full-data execution is currently blocked by dense-memory design and missing local Parquet inputs. See `docs/analysis_and_action_plan.md` and `docs/verification_matrix.md`.

## 0. Set up environment and paths

This notebook does not install Fractals, DeepTutor, Understand Anything or Learn Anything into the inference environment. They are development aids, not scientific dependencies; see `docs/third_party_tooling.md`. Kaggle inference must remain offline.

In [ ]:
%pip -q install numpy pandas pyarrow matplotlib seaborn scikit-learn rdkit==2026.3.3

# Colab normally includes PyTorch; local Jupyter should install requirements.txt before running.
import torch

from pathlib import Path
import json, sys, subprocess
import numpy as np
import pandas as pd

# Optional Google Drive mount. Uncomment and set DATA_DIR to your mounted directory.
# from google.colab import drive
# drive.mount('/content/drive')
DATA_DIR = Path('./data')  # set to your local/Drive folder containing train.parquet and test.parquet
OUTPUT_DIR = Path('./artifacts/casmi_workbench_v0.5.0')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Find extracted repository root (or set REPO explicitly if needed).
candidate_roots = [Path.cwd(), *Path.cwd().parents]
REPO = next((p for p in candidate_roots if (p / 'src' / 'identity.py').is_file()), None)
if REPO is None:
    raise FileNotFoundError('Extract the project zip and open this notebook from the extracted project, or set REPO to its root.')
sys.path.insert(0, str(REPO))
print({'repo': str(REPO), 'version': (REPO / 'VERSION').read_text().strip(), 'torch': torch.__version__, 'device': 'cuda' if torch.cuda.is_available() else 'cpu'})


## 1. Contract and data audit (streaming; no full train load)

Official documentation says one row per spectrum, with predictions per `molecule_id`. The scanner streams the large train parquet in Arrow batches and keeps only a bounded plotting/training sample. Test is small enough to read for EDA. The actual Arrow schema—not prose typos—is authoritative.

In [ ]:
from src.task1_eda import scan_parquet, plot_eda
from IPython.display import display, Image

TEST_PATH = DATA_DIR / 'test.parquet'
TRAIN_PATH = DATA_DIR / 'train.parquet'
if not TEST_PATH.is_file():
    raise FileNotFoundError(f'Missing test data: {TEST_PATH}. Download the Kaggle competition files locally; do not upload credentials here.')

# Set this lower for low-memory machines; bounded sample only affects plots and smoke modeling, not streamed counts.
SAMPLE_ROWS = 5000
reports, samples = {}, {}
if TRAIN_PATH.is_file():
    reports['train'], samples['train'] = scan_parquet(TRAIN_PATH, sample_rows=SAMPLE_ROWS, batch_size=5000, seed=42)
else:
    print('No train.parquet: test-only EDA is available, but supervised training is disabled.')
reports['test'], samples['test'] = scan_parquet(TEST_PATH, sample_rows=SAMPLE_ROWS, batch_size=5000, seed=42)
(OUTPUT_DIR / 'data_dictionary_and_quality.json').write_text(json.dumps(reports, indent=2, default=str))
plot_eda(samples.get('train'), samples.get('test'), OUTPUT_DIR / 'composite_eda.png')
display(pd.DataFrame({k: {'rows':v['rows'], 'unique_molecules':v['unique_molecules'], 'spectra_per_molecule_quantiles':v['spectra_per_molecule_quantiles'], 'sampled_rows':v['sample_rows']} for k,v in reports.items()}).T)
display(Image(filename=str(OUTPUT_DIR / 'composite_eda.png')))
display({k: v['columns'] for k,v in reports.items()})


### Interpret before proceeding

Check nulls, aligned peak arrays, spectrum/molecule cardinalities, adduct/mode/energy distributions, source and instrument imbalance. Kaggle's documentation enumerates ten test adducts but elsewhere says seven; the downloaded values and official schema win. The `spectra_per_molecule_distribution` and source counts in the JSON are exact scan summaries; precursor/peak quantiles use a bounded sample. Do not start training until you inspect these outputs.

In [ ]:
for name, report in reports.items():
    print(f'--- {name}: null counts ---')
    display(pd.Series(report['null_count']).sort_values(ascending=False).head(25))
    print(f'--- {name}: source/adduct/mode/instrument categories ---')
    display({k: v[:15] for k,v in report['category_counts'].items()})


## 2. Molecular identity and official MRR tests

Identity is evaluated after stereochemistry removal, tautomer canonicalization using RDKit 2026.3.3, then the first InChIKey block. Tests below are local unit tests; the Kaggle metric notebook remains the final authority.

In [ ]:
result = subprocess.run([sys.executable, '-m', 'unittest', 'discover', '-s', str(REPO / 'tests'), '-v'], cwd=REPO, text=True, capture_output=True)
print(result.stdout)
print(result.stderr)
if result.returncode != 0:
    raise RuntimeError('Unit tests failed; stop before modeling.')


## 3. Deterministic preprocessing / adduct smoke checks

Current encoding is a fixed 0.5 m/z binned intensity + square-root-intensity representation. This is a smoke baseline only—not an ablation-selected filter. Unknown adducts return NaN rather than silently treating m/z as neutral mass.

In [ ]:
from src.chemistry import neutral_mass
from src.solution import encode_spectrum, BIN_WIDTH, MAX_MZ

example = encode_spectrum([50.0, 100.0, 100.1, -1.0, float('nan')], [1.0, 0.25, 0.8, 0.3, 0.1])
assert example.shape == (2 * int(MAX_MZ / BIN_WIDTH),) and np.isfinite(example).all()
assert abs(neutral_mass(301.007276, '[M+H]+') - 300.0) < 1e-5
print({'encoded_length': len(example), 'nonzero_bins': int(np.count_nonzero(example)), 'neutral_mass_example': neutral_mass(301.007276, '[M+H]+')})


## 4. Optional bounded PyTorch smoke run (NOT a final submission)

The current prototype materializes dense spectra and is not accepted for the complete 2.5M-row dataset. Keep this off until the real data audit is reviewed. When enabled, it trains on at most `SAMPLE_ROWS` randomly sampled spectra, excludes any sampled training molecule IDs that also occur in the visible test preview, and writes `smoke_predictions.csv` only. This is an integration smoke test, not a validation score or competition submission.

To enable after the audit, set `RUN_SMOKE_MODEL = True`. For a final competition model, first replace this prototype with chunked training/retrieval and a valid identity-disjoint benchmark per the action plan.

In [ ]:
RUN_SMOKE_MODEL = False
if RUN_SMOKE_MODEL:
    if 'train' not in samples:
        raise RuntimeError('train.parquet is required for supervised modeling.')
    from src.solution import build_training, train_model, aggregate_test, rank_candidates, plot_diagnostics
    from argparse import Namespace
    train_sample = samples['train'].copy()
    test_frame = pd.read_parquet(TEST_PATH)
    if 'molecule_id' in train_sample and 'molecule_id' in test_frame:
        held_out_ids = set(test_frame.molecule_id.dropna().astype(str))
        train_sample = train_sample[~train_sample.molecule_id.astype(str).isin(held_out_ids)].copy()
    if len(train_sample) < 100:
        raise RuntimeError('Too few non-overlapping labeled spectra for smoke run; increase SAMPLE_ROWS or inspect test-preview overlap.')
    x, y, groups = build_training(train_sample, 'normalized_smiles')
    args = Namespace(seed=42, cpu=not torch.cuda.is_available(), validation_fraction=0.15, batch_size=32, learning_rate=1e-3, epochs=3)
    model, diagnostics = train_model(x, y, groups, args, OUTPUT_DIR)
    query_ids, query_x = aggregate_test(test_frame)
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    predictions = rank_candidates(model, query_ids, query_x, train_sample, test_frame, 'normalized_smiles', device, top_k=25)
    smoke = pd.DataFrame({'molecule_id': query_ids, 'smiles': predictions})
    smoke.to_csv(OUTPUT_DIR / 'smoke_predictions.csv', index=False)
    (OUTPUT_DIR / 'smoke_diagnostics.json').write_text(json.dumps(diagnostics, indent=2))
    print('SMOKE ONLY — do not submit. No MRR is available because hidden truth is absent.')
    display(smoke.head())
else:
    print('Smoke model disabled. Review the audit and versioned verification matrix first.')


## 5. Output, unresolved criteria, and next iteration

Outputs in `artifacts/casmi_workbench_v0.5.0/`: audit JSON, composite EDA plot, and optional smoke artifacts. **This workbench does not produce a competition-ready submission** until Tasks 3–8 pass their gates: full peak-cleaning experiments, leakage-safe spectral retrieval, candidate Recall@K, bounded-memory model/ranker, official MRR parity and offline <9 hour hidden-set rerun.

Follow `docs/analysis_and_action_plan.md` recursively: identify one measurable failure → state one hypothesis → plan a minimal experiment → implement/version → run tests and grouped evaluation → record result/negative result → update `docs/verification_matrix.md`. Keep `VERSION`, `CHANGELOG.md`, source/test headers synchronized.

For the next iteration, send the real `data_dictionary_and_quality.json` and EDA findings (not the full multi-GB parquet). Then we will lock schema/quality issues and design a leakage-safe retrieval baseline.